<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/Intellectual_property_UK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install required libraries
!pip install -q langchain langchain-community langchain-text-splitters sentence-transformers rank-bm25 faiss-cpu openai google-genai tiktoken pypdf beautifulsoup4


In [8]:
import os
import numpy as np
from typing import List
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss
from google.colab import userdata
import google.generativeai as genai

# Get Gemini API key from secrets and initialize
GoogleAIAPI = userdata.get('GoogleAIAPI')
genai.configure(api_key=GoogleAIAPI)
gemini_model = genai.GenerativeModel('gemini-2.5-flash') # Using gemini-pro as a default, you can change this if needed

In [22]:
# 1. Define your dynamic sources
web_urls = [
    # Real UK CDPA 1988 Legislation Links (Intro & Restricted Acts)
    "https://www.legislation.gov.uk/ukpga/1988/48/part/I/chapter/I",
    "https://www.legislation.gov.uk/ukpga/1988/48/part/I/chapter/II",
    # Recent UK IPO guidelines or BAILII court judgment URLs here
    "https://www.judiciary.uk/guidance-and-resources/judgment-summaries-for-the-commercial-court/",
    "https://www.judiciary.uk/guidance-and-resources/2025-judgment-summaries/"
]

# Recent judgments(March 2026) on intellectual property in the Music industry
pdf_paths = [
    "/content/Noel-Redding-Estate-Limited-v-Sony-Approved-Judgment-for-Handing-Down.pdf",
    "/content/data.pdf"
]

documents = []

# 2. Ingest Web URLs
print("Ingesting Web URLs...")
for url in web_urls:
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()
        documents.extend(docs)
        print(f"✅ Loaded: {url}")
    except Exception as e:
        print(f"❌ Failed to load {url}: {e}")

# 3. Ingest Local PDFs
print("\nIngesting PDFs...")
for path in pdf_paths:
    try:
        loader = PyPDFLoader(path)
        docs = loader.load()
        documents.extend(docs)
        print(f"✅ Loaded: {path}")
    except Exception as e:
        print(f"❌ Failed to load {path}: {e}")

print(f"\nTotal raw documents ingested: {len(documents)}")

Ingesting Web URLs...
✅ Loaded: https://www.legislation.gov.uk/ukpga/1988/48/part/I/chapter/I
✅ Loaded: https://www.legislation.gov.uk/ukpga/1988/48/part/I/chapter/II
✅ Loaded: https://www.judiciary.uk/guidance-and-resources/judgment-summaries-for-the-commercial-court/
✅ Loaded: https://www.judiciary.uk/guidance-and-resources/2025-judgment-summaries/

Ingesting PDFs...
✅ Loaded: /content/Noel-Redding-Estate-Limited-v-Sony-Approved-Judgment-for-Handing-Down.pdf
✅ Loaded: /content/data.pdf

Total raw documents ingested: 582


In [23]:
# Initialize the Recursive Character Text Splitter
# This preserves paragraph and sentence structures before breaking words
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # Slightly larger chunk for legal contexts
    chunk_overlap=100,    # Overlap to prevent cutting off crucial legal caveats
    separators=["\n\n", "\n", ".", " "]
)

# Split the loaded documents into manageable chunks
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from the ingested sources.")

# Preview the first chunk
if chunks:
    print("\n--- Preview of Chunk 1 ---")
    print(f"Source: {chunks[0].metadata.get('source', 'Unknown')}")
    print(chunks[0].page_content[:200] + "...")

Created 4540 chunks from the ingested sources.

--- Preview of Chunk 1 ---
Source: https://www.legislation.gov.uk/ukpga/1988/48/part/I/chapter/I
Copyright, Designs and Patents Act 1988...


In [24]:
print("Loading Embedding & Reranking Models (this may take a moment)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 1. Create Dense Vector Store (FAISS)
# We extract the text content from the LangChain Document objects
chunk_texts = [chunk.page_content for chunk in chunks]

print("Generating dense embeddings...")
embeddings = embedder.encode(chunk_texts)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

# 2. Create Sparse Index (BM25)
print("Generating sparse BM25 index...")
tokenized_corpus = [text.lower().split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Indexing complete.")

Loading Embedding & Reranking Models (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating dense embeddings...
Generating sparse BM25 index...
Indexing complete.


In [25]:
def reciprocal_rank_fusion(dense_ranks, sparse_ranks, k=60):
    rrf_scores = {}
    for rank, chunk_idx in enumerate(dense_ranks):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1 / (k + rank)
    for rank, chunk_idx in enumerate(sparse_ranks):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1 / (k + rank)

    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return sorted_indices

def retrieve_enhanced(query, top_k=4):
    """Hybrid Search + Cross-Encoder Reranking over Live Documents."""

    # 1. Dense Retrieval
    query_vector = embedder.encode([query])
    _, dense_indices = index.search(np.array(query_vector).astype('float32'), top_k*3)
    dense_indices = dense_indices[0].tolist()

    # 2. Sparse Retrieval
    tokenized_query = query.lower().split()
    sparse_scores = bm25.get_scores(tokenized_query)
    sparse_indices = np.argsort(sparse_scores)[::-1][:top_k*3].tolist()

    # 3. Fuse Results (RRF)
    fused_indices = reciprocal_rank_fusion(dense_indices, sparse_indices)[:top_k*2]
    candidate_chunks = [chunks[i] for i in fused_indices]

    # 4. Cross-Encoder Reranking
    cross_inp = [[query, chunk.page_content] for chunk in candidate_chunks]
    cross_scores = reranker.predict(cross_inp)

    # Sort by reranker score
    scored_chunks = list(zip(candidate_chunks, cross_scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, score in scored_chunks[:top_k]]

def generate_answer(query, retrieved_chunks):
    """Generates the grounded response referencing dynamic metadata."""

    # Inject both text AND the source metadata into the prompt
    context_blocks = []
    for c in retrieved_chunks:
        source = c.metadata.get('source', 'Unknown Source')
        page = c.metadata.get('page', '')
        page_str = f" (Page {page})" if page else ""
        context_blocks.append(f"Source: {source}{page_str}\n{c.page_content}")

    context_string = "\n\n---\n\n".join(context_blocks)

    system_prompt = (
        "You are an expert UK Music IP Lawyer. Answer the user's query based ONLY on the provided legal text. "
        "You must explicitly cite the Source URLs or File Names provided in the context in your response. "
        "If the answer is not contained in the context, say 'I cannot advise based on the currently ingested documents.'"
    )

    # Using gemini_model instead of client
    full_prompt = f"System: {system_prompt}\nUser: Context:\n{context_string}\n\nQuery: {query}"
    response = gemini_model.generate_content(full_prompt)
    return response.text

In [26]:
# Let's test the pipeline against the actual UK legislation we just ingested!
query = "What are the specific acts restricted by copyright according to the CDPA 1988?"

print(f"Query: {query}\n")
print("Retrieving context from ingested web data...\n")

# Retrieve and Generate
retrieved_docs = retrieve_enhanced(query)
answer = generate_answer(query, retrieved_docs)

print("⚖️ AI Lawyer Response:\n")
print(answer)

print("\n🔍 Cited Sources used for this answer:")
for doc in retrieved_docs:
    print(f"- {doc.metadata.get('source')}")

Query: What are the specific acts restricted by copyright according to the CDPA 1988?

Retrieving context from ingested web data...

⚖️ AI Lawyer Response:

According to the Copyright, Designs and Patents Act 1988, the owner of copyright in a work has the exclusive right to do the following acts in the United Kingdom:
*   to copy the work (see section 17)
*   to issue copies of the work to the public (see section 18)
*   to rent or lend the work to the public (see section 18A)
*   to perform, show or play the work in public (see section 19)
*   to communicate the work to the public (see section 20)

Source: /content/data.pdf (Page 16)

🔍 Cited Sources used for this answer:
- /content/data.pdf
- /content/data.pdf
- /content/data.pdf
- /content/data.pdf


In [34]:
# Testing the pipeline with more questions
def run_query(query):
    print(f"Query: {query}\n")
    print("Retrieving context from ingested web data...\n")

    # Retrieve and Generate
    retrieved_docs = retrieve_enhanced(query)
    answer = generate_answer(query, retrieved_docs)

    print("⚖️ AI Lawyer Response:\n")
    print(answer)

    print("\n🔍 Cited Sources used for this answer:")
    for doc in retrieved_docs:
        print(f"- {doc.metadata.get('source')}")

# Example usage with the previous query
run_query("What does intellectual property mean in music industry")
run_query("When is the latest ruling concerning music industry")

Query: What does intellectual property mean in music industry

Retrieving context from ingested web data...

⚖️ AI Lawyer Response:

In the music industry, intellectual property rights, such as copyright and performers' rights, comprise a bundle of legal rights. Each of these rights corresponds to the ability to authorise or prohibit particular acts concerning the musical work or performance. These rights are dynamic and can expand to cover new uses, such as the introduction of the "making available right" to accommodate the digital exploitation of music (Source: /content/Noel-Redding-Estate-Limited-v-Sony-Approved-Judgment-for-Handing-Down.pdf, Page 83).

The benefits derived from intellectual property serve as an incentive for innovation and creativity. For instance, a person who conceives a good idea, such as composing a significant musical piece, is entitled to protect the advantage gained from this creation (Source: /content/Noel-Redding-Estate-Limited-v-Sony-Approved-Judgment-for